# Занятие 9, пара 1. Аугментации и разбиение без утечек


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Больше картинок — не всегда больше независимых данных

В 08 мы сохранили метки и группы происхождения. Теперь решим, какие изображения можно использовать
для обучения, какие — для выбора метода, а какие — для итоговой проверки.
Затем добавим изменения изображений так, чтобы они не подменили задачу и не испортили оценку.

После пары вы сможете объяснить допустимость аугментации, разделить **группы**, проверить пересечения
и сохранить список разбиения. Основные примеры — NumPy/Matplotlib и стандартный Python, без OpenCV.
Будем разбирать смысл действий и их последствия; функции и циклы Python уже знакомы.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


### 1. Что хотим сделать устойчивым

**Аугментация** — создание изменённого варианта обучающего примера. Мы предполагаем, что изменение
возможно в будущих данных и не разрушает нужный ответ. Для класса «чашка» небольшое изменение яркости
обычно не меняет метку. Для задачи «оценить экспозицию кадра» оно меняет сам ответ.
Поэтому список допустимых преобразований начинается с постановки задачи.

Откроем крупную фотографию из предыдущего блока. Переведём uint8 в float 0…1 до умножения:
переполнение целого типа не должно стать случайной «аугментацией». `np.clip` ограничит значения диапазоном.


In [ ]:
import csv
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

photo = np.load(DATA_DIR / 'segmentation' / 'coffee.npy').astype(float) / 255
brighter = np.clip(photo * 1.3, 0, 1)
darker = np.clip(photo * 0.6, 0, 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, picture, title in zip(axes, [photo, darker, brighter], ['Исходник', 'Темнее', 'Светлее']):
    ax.imshow(picture)
    ax.set_title(title)
    ax.axis('off')
plt.show()


Это умножение RGB — упрощённая имитация изменения яркости, не полная физическая модель камеры.
Пересвет обрезает значения: из насыщенного белого нельзя восстановить исходную деталь.
Сильное преобразование может сделать пример неинформативным, хотя формальная метка не изменилась.

### 2. Геометрия: изменяем картинку и ответ вместе

`photo[:, ::-1]` оставляет строки в прежнем порядке, а столбцы читает справа налево — отражение по горизонтали.
Для классификации чашки метка остаётся той же. Для рамки меняются координаты.
Если ширина кадра W, рамка по границам `[x1,y1,x2,y2]` переходит в `[W−x2,y1,W−x1,y2]`.
Используем соглашение о границах из 08, поэтому здесь **W**, а не W−1; W−1 относится к индексам отдельных пикселей.


In [ ]:
flipped = photo[:, ::-1]
height, width = photo.shape[:2]
box = [172, 18, 412, 310]
x1, y1, x2, y2 = box
flipped_box = [width-x2, y1, width-x1, y2]
print('До:', box, 'после:', flipped_box)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, picture, current_box, title in zip(axes, [photo, flipped], [box, flipped_box], ['До', 'После']):
    x1, y1, x2, y2 = current_box
    ax.imshow(picture)
    ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor='lime',linewidth=3))
    ax.set_title(title)
plt.show()


Проверка «рамка внутри изображения» недостаточна: забытая старая рамка тоже может остаться внутри.
Нужна проверка соответствия содержимому. Для маски применяют то же отражение, для ключевых точек
пересчитывают координаты; при кадрировании ещё решают, не исчез ли целевой объект.
В проекте классификации рамки не обязательны, но принцип согласованности общий.

### 3. Когда привычный список аугментаций ломает класс

Рассмотрим **синтетический** знак со стрелкой. Сгенерируем крупное изображение 300×500.
`np.indices((h,w))` даёт таблицы координат y и x. Прямоугольная маска задаёт хвост,
неравенство по расстоянию от средней строки — треугольный наконечник.
Это предоставленный геометрический пример для рассуждения, не алгоритм распознавания знаков.


In [ ]:
yy, xx = np.indices((300, 500))
stem = (xx >= 80) & (xx <= 330) & (yy >= 130) & (yy <= 170)
head = (xx >= 300) & (xx <= 420) & (np.abs(yy-150) <= (420-xx)*0.6)
arrow = np.ones((300, 500, 3), dtype=float)
arrow[stem | head] = [0.05, 0.25, 0.55]
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(arrow)
ax.set_title('Новая задача: определить направление стрелки')
ax.axis('off')
plt.show()


### ✏️ Ваш ход — Сохранить метку или заменить?

Классы теперь `'left'` и `'right'`. Для исходного знака метка right.
Сравните горизонтальное отражение `arrow[:, ::-1]`, вертикальное `arrow[::-1]` и поворот на 90°
через `np.rot90(arrow)` (готовая операция поворота массива против часовой стрелки).

Создайте `label_policy`: ключи `'horizontal'`, `'vertical'`, `'rotate90'`; значения — корректная метка
либо None, если пример выходит за наши два класса. Покажите преобразования и объясните,
почему единый ответ «аугментации сохраняют метку» здесь не работает.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('09.1.1', label_policy)


**Мой вывод:** …


### 4. Сначала разбиение, потом новые варианты train

**Train** используется для обучения. **Validation** — для выбора признаков и параметров.
**Test** — отложенная итоговая проверка после фиксации решения.
Если отражение одной картинки попадёт в train, а оригинал — в test, модель увидит почти знакомый пример.
Точность может оказаться высокой, но она плохо отвечает вопросу о новых объектах.

Правильный порядок: определить независимую группу → разделить группы → аугментировать только train.
На validation/test допускается заранее фиксированная общая подготовка (размер, формат, масштаб),
но не случайная выдача новых версий при каждом измерении. Если оцениваем устойчивость к искажениям,
делаем отдельный явно описанный тест, не подменяем основной.

На знакомых синтетических обложках одна группа — один объект в двух вариантах яркости.
Используем их здесь **для проверки устройства разбиения**, не для новой оценки модели: этот набор мы уже видели.


In [ ]:
covers = np.load(DATA_DIR / 'segmentation' / 'covers.npz')
images = covers['images']
labels = covers['labels']
groups = covers['object_ids']
print('Изображений:', len(images), 'объектов:', len(np.unique(groups)))
first_group = groups[0]
first_indices = np.flatnonzero(groups == first_group)
fig, axes = plt.subplots(1, len(first_indices), figsize=(10, 4))
for ax, index in zip(axes, first_indices):
    ax.imshow(images[index])
    ax.set_title(f'Одна группа: {groups[index]}')
plt.show()


Случайное перемешивание строк не понимает связь изображений. Сначала выделим уникальные группы.
У нашего учебного объекта все кадры одного класса; проверим это, а не предположим молча.
В реальном видео могут быть разные классы внутри одной группы — тогда следующий простой способ
балансировки по классу не подходит. Групповая независимость важнее красивой ровной пропорции.


In [ ]:
unique_groups = np.unique(groups)
group_labels = {}
for group in unique_groups:
    current_labels = np.unique(labels[groups == group])
    assert len(current_labels) == 1
    group_labels[group] = int(current_labels[0])
print(group_labels)


### 5. Разделим группы и сохраним решение

Для каждого из двух классов у нас семь групп. Четыре отправим в train, одну в validation, две в test.
Это конкретный учебный расклад, а не универсальная пропорция для любого проекта.
`rng.permutation` перемешивает список; фиксированный seed воспроизводит порядок.
Не перебираем seed до получения самого удобного теста.

Сначала увидим распределение групп одного класса, затем повторим то же для остальных.


In [ ]:
rng = np.random.default_rng(42)
class_zero_groups = []
for group in unique_groups:
    if group_labels[group] == 0:
        class_zero_groups.append(group)
shuffled_zero = rng.permutation(class_zero_groups)
print('train:', shuffled_zero[:4])
print('validation:', shuffled_zero[4:5])
print('test:', shuffled_zero[5:])


Теперь общий словарь `split_by_group`. Заново создаём генератор с тем же seed,
чтобы результат не зависел от количества пробных запусков предыдущей ячейки.
В каждую группу записываем **одну** часть, а потом переносим её на все изображения группы.


In [ ]:
rng = np.random.default_rng(42)
split_by_group = {}
for class_id in np.unique(labels):
    candidates = []
    for group in unique_groups:
        if group_labels[group] == class_id:
            candidates.append(group)
    assert len(candidates) == 7
    shuffled = rng.permutation(candidates)
    for group in shuffled[:4]:
        split_by_group[group] = 'train'
    for group in shuffled[4:5]:
        split_by_group[group] = 'validation'
    for group in shuffled[5:]:
        split_by_group[group] = 'test'

split = np.array([split_by_group[group] for group in groups])
for part in ['train', 'validation', 'test']:
    mask = split == part
    print(part, 'изображений:', mask.sum(), 'групп:', len(np.unique(groups[mask])),
          'классы:', np.unique(labels[mask]))


Сам seed в отчёте недостаточен: порядок исходных строк и версия набора тоже влияют на разбиение.
Поэтому сохранение явного списка полезнее обещания «мы когда-то перемешали с 42».
Но сначала проверим, правильно ли вообще выбрана группа. Связь может идти и через объект,
и через серию съёмки. Тогда запрет на пересечение передаётся по цепочке.


### ✏️ Ваш ход — Найти косвенную утечку

Четыре кадра:

| Кадр | Объект | Серия |
|---|---|---|
| A | cup_1 | video_1 |
| B | cup_2 | video_1 |
| C | cup_2 | video_2 |
| D | spoon_1 | video_3 |

В этом эксперименте ни объект, ни серия не должны пересекать части.
Запишите `merged_groups`: словарь кадр → номер объединённой группы.
Какие кадры надо держать вместе, даже если у них нет прямого совпадения обоих полей?
Можно ли на этих четырёх кадрах честно получить три непустые независимые части?


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('09.1.2', merged_groups)


**Мой вывод:** …


### 6. Проверки и перенос аугментаций

Пересечение множеств group_id должно быть пустым для каждой пары частей.
Проверим также, что каждый кадр получил допустимую часть. После этого покажем один вариант train.
Его метка и group_id остаются прежними; меняется только содержимое изображения.


In [ ]:
train_groups = set(groups[split == 'train'])
validation_groups = set(groups[split == 'validation'])
test_groups = set(groups[split == 'test'])
assert not train_groups & validation_groups
assert not train_groups & test_groups
assert not validation_groups & test_groups
assert set(split) == {'train', 'validation', 'test'}
train_index = int(np.flatnonzero(split == 'train')[0])
train_image = images[train_index].astype(float) / 255
augmented_train = np.clip(train_image * 0.8, 0, 1)
print('Класс:', labels[train_index], 'группа:', groups[train_index], 'часть:', split[train_index])


**Хеш файла не доказывает независимость.** `hashlib.sha256(байты).hexdigest()` даёт короткую запись
для точного сравнения содержимого. Даже небольшое изменение яркости меняет хеш,
но объект от этого не становится другим. `.tobytes()` отдаёт байты массива.
Для точного сравнения формата надо учитывать ещё shape и dtype; ниже оба массива одинакового формата.


In [ ]:
original_hash = hashlib.sha256(train_image.tobytes()).hexdigest()
changed_hash = hashlib.sha256(augmented_train.tobytes()).hexdigest()
print('Хеши равны:', original_hash == changed_hash)
print('Физический объект всё ещё тот же:', groups[train_index])


### ✏️ Ваш ход — Отличающиеся файлы прошли проверку

Коллега создал оригинал и затемнённую копию каждого кадра, присвоил им разные имена,
проверил, что SHA-256 различаются, и случайно разделил все файлы.

Напишите `audit_augmented(records)`: вход — список словарей с полями group_id и split,
выход — множество group_id, попавших более чем в одну часть. Имена и хеши для этой проверки не нужны.
Набросайте контрпример из двух файлов одного объекта в разных частях и правильный вариант.
Группа — известное происхождение, а не номер, заново выданный каждой аугментации.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('09.1.3', audit_augmented)


**Мой вывод:** …


### Сохраним список разбиения

Основная ячейка ниже не использует ответы упражнений. Индекс ссылается на строку исходного covers.npz;
в своём проекте обычно сохраняют относительный путь, метку и group_id.
`csv.DictWriter` знаком по 08.1. Этот файл фиксирует учебное разбиение, а не меняет исходный набор.


In [ ]:
output_dir = WORKSPACE_DIR / 'outputs' / 'split_practice'
output_dir.mkdir(parents=True, exist_ok=True)
with (output_dir / 'split.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=['index', 'label', 'group_id', 'split'])
    writer.writeheader()
    for index in range(len(images)):
        writer.writerow({'index':index, 'label':int(labels[index]),
                         'group_id':str(groups[index]), 'split':str(split[index])})
print(output_dir / 'split.csv')


## Что перенести в ЛР4

Определите независимую группу для своего проекта, сохраните разбиение и проверьте пересечения.
Запишите допустимые аугментации и причину, почему они сохраняют нужный класс. Не аугментируйте всё заранее
до разделения. Фиксированные преобразования подготовки тоже должны быть описаны.

**Резерв 10–15 минут:** найдите почти одинаковую пару изображений из одного объекта и сравните среднюю
абсолютную разность пикселей. Почему малое расстояние — повод проверить пару, но не доказательство общего происхождения?
Перцептивные хеши помогают искать кандидатов, но не заменяют историю съёмки и ручную проверку.

В больших проектах используют [GroupShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html)
и похожие инструменты. Они разделяют переданные группы, но не умеют угадывать правильный group_id.
Следующая пара: обучим простой классификатор и сохраним test для итоговой оценки.
